# BPIC17 vs Simulated Log Comparison

This notebook compares the original BPIC 2017 event log against a simulated event log and quantifies how well simulation approximates reality.

It focuses on:
- activity distribution similarity
- trace variant overlap
- directly-follows (DFG) overlap
- case/event counts and throughput characteristics
- start/end activity alignment

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / "integration").exists():
    repo_root = cwd
elif (cwd.parent / "integration").exists():
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from integration.SimulationBenchmark import SimulationBenchmark

print("Repo root:", repo_root)

In [ ]:
# Configure paths
ORIGINAL_LOG_PATH = repo_root / "Dataset" / "BPI Challenge 2017.xes"
SIMULATED_LOG_PATH = repo_root / "integration" / "output" / "simulated_log.csv"

print("Original log:", ORIGINAL_LOG_PATH)
print("Simulated log:", SIMULATED_LOG_PATH)

if not ORIGINAL_LOG_PATH.exists():
    raise FileNotFoundError(f"Original log not found: {ORIGINAL_LOG_PATH}")
if not SIMULATED_LOG_PATH.exists():
    raise FileNotFoundError(f"Simulated log not found: {SIMULATED_LOG_PATH}")

In [ ]:
# Build benchmark (set filter_lifecycle_complete=True if you want complete-only comparison)
benchmark = SimulationBenchmark(
    original_log=str(ORIGINAL_LOG_PATH),
    simulated_log=str(SIMULATED_LOG_PATH),
    filter_lifecycle_complete=False,
)

results = benchmark.compute_all_metrics()
print("Computed benchmark sections:", list(results.keys()))

In [ ]:
# Quick headline summary table
results["simple_metrics"]

In [ ]:
# Core descriptive tables
results["basic_stats"]

In [ ]:
results["events_per_case"]

In [ ]:
results["throughput_time"]

In [ ]:
# Activity distribution comparison (top-20 by original frequency)
activity_cmp = results["activity_distribution"].copy()
activity_top20 = activity_cmp.sort_values("Original Count", ascending=False).head(20)

x = np.arange(len(activity_top20))
width = 0.42

plt.figure(figsize=(16, 6))
plt.bar(x - width/2, activity_top20["Original Share (%)"], width=width, label="Original")
plt.bar(x + width/2, activity_top20["Simulated Share (%)"], width=width, label="Simulated")
plt.xticks(x, activity_top20["Activity"], rotation=70, ha="right")
plt.ylabel("Share (%)")
plt.title("Top-20 Activity Distribution: Original vs Simulated")
plt.legend()
plt.tight_layout()
plt.show()

activity_top20[["Activity", "Original Share (%)", "Simulated Share (%)", "Share Difference (%)"]]

In [ ]:
# Trace variant alignment
variants_cmp = results["variants_comparison"].copy()
variants_cmp.head(20)

In [ ]:
# DFG edge alignment
results["dfg_comparison"].head(25)

In [ ]:
# Start and end activities
print("Start activities")
display(results["start_activities"].head(20))

print("End activities")
display(results["end_activities"].head(20))

## Interpreting Results

Use these signals as a quick approximation scorecard:
- **Activity Jaccard Similarity**: higher means better activity coverage match.
- **Trace Variant Overlap**: higher means generated process paths resemble original paths.
- **DFG Edge Jaccard Similarity**: higher means local control-flow transitions are preserved.
- **Count Ratios (cases/events)** near 100%: simulation scale matches source log.
- **Share Difference (%)** near 0 for frequent activities: good distribution calibration.

Optional: export all benchmark tables to Excel in the next cell.

In [ ]:
# Optional export
OUT_XLSX = repo_root / "integration" / "output" / "bpic17_vs_simulation_benchmark.xlsx"
benchmark.export_results(str(OUT_XLSX))
print("Exported:", OUT_XLSX)